In [2]:
import sys
import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc
import numpy as np

class NavierStokesSolver:
    """
    Solves the time-dependent Navier-Stokes equations in 2D using DMPlex.
    Translated from PETSc C example ex46.c.
    """
    def __init__(self, mms_type=1, reynolds=400.0):
        self.mms_type = mms_type
        self.re = reynolds
        self.dm = None
        self.ts = None

    def create_mesh(self):
        """Creates a parallel unstructured mesh (DMPLEX)."""
        self.dm = PETSc.DMPlex().createBoxMesh(
            faces=[4, 4], 
            lower=[0, 0], 
            upper=[1, 1], 
            simplex=False, 
            interpolate=True
        )
        self.dm.setFromOptions()
        self.dm.viewFromOptions("-dm_view")

    def setup_discretization(self):
        """Sets up Finite Element spaces for velocity and pressure."""
        # P2 elements for velocity, P1 for pressure (Taylor-Hood)
        fe_vel = PETSc.FE().createDefault(self.dm.getDimension(), self.dm.getDimension(), 
                                          False, "vel_", PETSC_DEFAULT)
        fe_pres = PETSc.FE().createDefault(self.dm.getDimension(), 1, 
                                          False, "pres_", PETSC_DEFAULT)
        
        self.dm.setField(0, fe_vel)
        self.dm.setField(1, fe_pres)
        self.dm.createDS()

    def mms_solutions(self, t, x):
        """
        Implementation of MMS1 from ex46.c
        u = t + x^2 + y^2; v = t + 2x^2 - 2xy; p = x + y - 1
        """
        if self.mms_type == 1:
            u = t + x[0]**2 + x[1]**2
            v = t + 2.0*x[0]**2 - 2.0*x[0]*x[1]
            p = x[0] + x[1] - 1.0
            return [u, v], [p]

    def solve(self, max_steps=10, dt=0.1):
        """Configures the TS (Time Stepping) solver."""
        self.ts = PETSc.TS().create(PETSc.COMM_WORLD)
        self.ts.setDM(self.dm)
        self.ts.setProblemType(self.ts.ProblemType.NONLINEAR)
        
        # In petsc4py, we often use the DMPlexTS helpers 
        # to connect the C residual functions via the DS
        self.ts.setIFunction(PETSc.DMPlex().computeIFunctionFEM, self.dm)
        self.ts.setIJacobian(PETSc.DMPlex().computeIJacobianFEM, self.dm)
        
        self.ts.setTimeStep(dt)
        self.ts.setMaxSteps(max_steps)
        self.ts.setExactFinalTime(self.ts.ExactFinalTime.STEPOVER)
        self.ts.setFromOptions()
        
        # Initial condition
        u_init = self.dm.createGlobalVec()
        # Project initial MMS solution at t=0
        # (Simplified projection logic for brevity)
        
        self.ts.solve(u_init)
        return u_init

if __name__ == "__main__":
    solver = NavierStokesSolver()
    solver.create_mesh()
    solver.setup_discretization()
    solution = solver.solve()
    print("Flow simulation complete.")

Error: error code 63
[0] DMPlexCreateBoxMesh() at /private/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/pip-install-x61f0lna/petsc_f95662ad7c7d4869b7c3f62ceaf107d5/src/dm/impls/plex/plexcreate.c:2037
[0] DMPlexCreateBoxMesh_Internal() at /private/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/pip-install-x61f0lna/petsc_f95662ad7c7d4869b7c3f62ceaf107d5/src/dm/impls/plex/plexcreate.c:1948
[0] DMPlexCreateBoxMesh_Simplex_Internal() at /private/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/pip-install-x61f0lna/petsc_f95662ad7c7d4869b7c3f62ceaf107d5/src/dm/impls/plex/plexcreate.c:1473
[0] DMPlexGenerate() at /private/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/pip-install-x61f0lna/petsc_f95662ad7c7d4869b7c3f62ceaf107d5/src/dm/impls/plex/plexgenerate.c:202
[0] Argument out of range
[0] No grid generator of dimension 2 registered You may need to add --download-triangle to your ./configure options